# 06. Sistemas discretos, respuesta al impulso y convolución

**Objetivos de aprendizaje.** Al terminar podrás:

1. Representar un sistema como una **función de Python** que recibe $x[n]$ y regresa $y[n]$.
2. Comprobar numéricamente si un sistema es **lineal**, **invariante en el tiempo**, **causal** y **estable**.
3. Calcular la salida de un sistema descrito por una **ecuación en diferencias**.
4. Obtener la **respuesta al impulso** $h[n]$ y distinguir sistemas **FIR** e **IIR**.
5. Entender la **convolución** como consecuencia de "toda señal es una suma de impulsos", calcularla a mano y con código,
   **respetando los índices**.
6. Usar sistemas para **filtrar** señales reales (ECG, audio).

**Requisito:** notebooks 02 y 05.

In [ ]:
# Celda de preparación. Ejecútala antes que cualquier otra (Shift + Enter).
import sys
import os

# Si estás en Google Colab, esta parte descarga el material del curso.
# En tu computadora no hace nada.
URL_REPO = "https://github.com/TU_USUARIO/TU_REPOSITORIO.git"
if "google.colab" in sys.modules and not os.path.exists("../funciones_pds"):
    if not os.path.exists("/content/pds"):
        os.system(f"git clone -q {URL_REPO} /content/pds")
    os.chdir("/content/pds/notebooks")   # así funcionan las rutas "../datos/..."

sys.path.append("..")            # para que Python encuentre la carpeta funciones_pds

import numpy as np               # cálculos con arreglos
import matplotlib.pyplot as plt  # gráficas
import funciones_pds as pds      # la biblioteca del curso

plt.rcParams["figure.figsize"] = (9, 3)   # tamaño por defecto de las figuras
print("Bibliotecas cargadas. Ya puedes continuar.")

## 1. ¿Qué es un sistema?

Un sistema es cualquier proceso que transforma una señal de entrada en una de salida:

$$x[n] \;\longrightarrow\; \boxed{\;\mathcal{T}\;} \;\longrightarrow\; y[n] = \mathcal{T}\{x[n]\}$$

En Python, un sistema es simplemente una **función** que recibe un arreglo y regresa otro.
(En este notebook, las señales empiezan en $n=0$ y suponemos $x[n]=0$ para $n<0$.)

In [ ]:
def amplificador(x):
    '''y[n] = 3·x[n]'''
    y = 3 * x
    return y


def retardo_2(x):
    '''y[n] = x[n-2]: agrega 2 ceros al inicio y quita las 2 últimas muestras'''
    y = np.zeros(len(x))
    for n in range(len(x)):
        if n - 2 >= 0:
            y[n] = x[n - 2]
    return y


def cuadrado(x):
    '''y[n] = x[n]²'''
    y = x ** 2
    return y


def multiplica_por_n(x):
    '''y[n] = n·x[n]'''
    n = np.arange(len(x))
    y = n * x
    return y


def acumulador(x):
    '''y[n] = x[0] + x[1] + ... + x[n]'''
    y = np.zeros(len(x))
    suma = 0.0
    for n in range(len(x)):
        suma = suma + x[n]
        y[n] = suma
    return y

In [ ]:
x = np.array([1.0, 2.0, 0.0, -1.0, 0.0, 0.0])
print("x               =", x)
print("amplificador    =", amplificador(x))
print("retardo_2       =", retardo_2(x))
print("cuadrado        =", cuadrado(x))
print("multiplica_por_n=", multiplica_por_n(x))
print("acumulador      =", acumulador(x))

## 2. Propiedades de los sistemas

| Propiedad | Definición | Prueba numérica |
|---|---|---|
| **Linealidad** | $\mathcal{T}\{a x_1 + b x_2\} = a\mathcal{T}\{x_1\} + b\mathcal{T}\{x_2\}$ | `pds.probar_linealidad` |
| **Invarianza en el tiempo** | si $x[n] \to y[n]$ entonces $x[n-k] \to y[n-k]$ | `pds.probar_invarianza` |
| **Causalidad** | $y[n]$ sólo depende de $x[n], x[n-1], \dots$ (no del futuro) | inspección |
| **Estabilidad (BIBO)** | entrada acotada → salida acotada | $\sum \lvert h[n]\rvert < \infty$ |

> Ojo: una prueba numérica **puede demostrar que un sistema NO cumple** una propiedad (basta un contraejemplo),
> pero **no demuestra** que sí la cumple para todas las señales. Para eso se necesita la demostración matemática.

In [ ]:
import inspect
print(inspect.getsource(pds.probar_linealidad))

In [ ]:
x1 = np.random.randn(20)
x2 = np.random.randn(20)

sistemas = [amplificador, retardo_2, cuadrado, multiplica_por_n, acumulador]
for sistema in sistemas:
    print(f"--- {sistema.__name__} ---")
    lineal = pds.probar_linealidad(sistema, x1, x2)
    invariante = pds.probar_invarianza(sistema, x1)
    print(f"    lineal: {lineal}   invariante: {invariante}")

Los sistemas que son **Lineales e Invariantes en el Tiempo (LTI)** son los más importantes, porque
quedan **completamente descritos por su respuesta al impulso**. Veamos por qué.

## 3. Ecuación en diferencias

La mayoría de los sistemas LTI prácticos se describen con una **ecuación en diferencias**:

$$y[n] = \sum_{k=0}^{M} b_k\, x[n-k] \;-\; \sum_{k=1}^{K} a_k\, y[n-k]$$

### Ejemplo a mano: $y[n] = x[n] + 0.5\,y[n-1]$ con $x[n] = \delta[n]$

| $n$ | $x[n]$ | $y[n-1]$ | $y[n] = x[n] + 0.5\,y[n-1]$ |
|---|---|---|---|
| 0 | 1 | 0 (reposo) | $1 + 0 = 1$ |
| 1 | 0 | 1 | $0 + 0.5 = 0.5$ |
| 2 | 0 | 0.5 | $0 + 0.25 = 0.25$ |
| 3 | 0 | 0.25 | $0.125$ |

El mismo cálculo en código, paso a paso:

In [ ]:
N = 6
x = np.zeros(N)
x[0] = 1.0                        # impulso
y = np.zeros(N)

for n in range(N):
    if n - 1 >= 0:
        y_anterior = y[n - 1]
    else:
        y_anterior = 0.0          # el sistema empieza en reposo
    y[n] = x[n] + 0.5 * y_anterior
    print(f"n = {n}: y[n] = {x[n]} + 0.5·{y_anterior} = {y[n]}")

Con la forma general $b = [b_0, b_1, \dots]$, $a = [1, a_1, \dots]$. **Cuidado con el signo.**
$y[n] = x[n] + 0.5\,y[n-1]$ se escribe como $y[n] - 0.5\,y[n-1] = x[n]$, entonces $a = [1, -0.5]$.

In [ ]:
print(inspect.getsource(pds.ecuacion_diferencias))

In [ ]:
y = pds.ecuacion_diferencias(b=[1.0], a=[1.0, -0.5], x=x)
print(y)

## 4. Respuesta al impulso $h[n]$

Es la salida del sistema cuando la entrada es $\delta[n]$. Hay dos tipos:

- **FIR** (*Finite Impulse Response*): $h[n]$ dura un número finito de muestras. Sin realimentación ($a = [1]$).
- **IIR** (*Infinite Impulse Response*): $h[n]$ dura para siempre. Tiene realimentación (usa salidas pasadas).

In [ ]:
# FIR: promedio móvil de 4 muestras  y[n] = (x[n] + x[n-1] + x[n-2] + x[n-3]) / 4
n, h_fir = pds.respuesta_al_impulso(b=[0.25, 0.25, 0.25, 0.25], a=[1], N=15)

# IIR: y[n] = x[n] + 0.8 y[n-1]
n, h_iir = pds.respuesta_al_impulso(b=[1], a=[1, -0.8], N=15)

figura, ejes = plt.subplots(1, 2, figsize=(13, 3))
pds.graficar_discreta(n, h_fir, titulo="FIR: promedio móvil (4 muestras)", etiqueta_y="h[n]", ax=ejes[0])
pds.graficar_discreta(n, h_iir, titulo="IIR: h[n] = 0.8ⁿ u[n]", etiqueta_y="h[n]", ax=ejes[1])
plt.tight_layout()
plt.show()

## 5. Convolución: por qué $y = x * h$

Juntemos tres ideas:

1. **Toda señal es suma de impulsos** (notebook 02): $\;x[n] = \sum_k x[k]\,\delta[n-k]$
2. **Invarianza**: si $\delta[n] \to h[n]$, entonces $\delta[n-k] \to h[n-k]$
3. **Linealidad**: si $\delta[n-k] \to h[n-k]$, entonces $x[k]\,\delta[n-k] \to x[k]\,h[n-k]$, y la salida de una suma es la suma de salidas.

$$\boxed{\,y[n] = \sum_{k} x[k]\,h[n-k] = x[n] * h[n]\,}$$

### 5.1 Verlo: cada muestra de $x$ "dispara" una copia de $h$

Ejemplo: $x[n] = \{\underset{\uparrow}{1}, 2, 1\}$, $\;h[n] = \{\underset{\uparrow}{1}, 1, 1\}$ (ambas empiezan en $n=0$).

In [ ]:
x = np.array([1.0, 2.0, 1.0])
h = np.array([1.0, 1.0, 1.0])
Ly = len(x) + len(h) - 1                # longitud de la salida
n_y = np.arange(0, Ly)

copias = []
for k in range(len(x)):
    copia = np.zeros(Ly)
    for j in range(len(h)):
        copia[k + j] = x[k] * h[j]      # h desplazada k muestras y escalada por x[k]
    copias.append(copia)
    print(f"x[{k}]·h[n-{k}] = {copia}")

y = np.zeros(Ly)
for copia in copias:
    y = y + copia
print("suma          =", y)

figura, ejes = plt.subplots(1, len(copias) + 1, figsize=(15, 3), sharey=True)
for k in range(len(copias)):
    pds.graficar_discreta(n_y, copias[k], titulo=f"x[{k}]·h[n-{k}]", ax=ejes[k])
pds.graficar_discreta(n_y, y, titulo="y[n] = suma", ax=ejes[-1], color="C3")
plt.tight_layout()
plt.show()

### 5.2 Calcularlo con la fórmula: "invertir, desplazar, multiplicar y sumar"

Para cada $n$: $\;y[n] = \sum_k x[k]\,h[n-k]$. Como función de $k$, $h[n-k]$ es $h$ **invertida** y **desplazada** $n$ lugares.
Para cada $n$ multiplicamos $x[k]$ por $h[n-k]$ muestra a muestra y sumamos:

In [ ]:
for n in range(Ly):
    terminos = []
    suma = 0.0
    for k in range(len(x)):
        j = n - k
        if 0 <= j < len(h):
            terminos.append(f"x[{k}]·h[{j}]")
            suma = suma + x[k] * h[j]
    print(f"y[{n}] = " + " + ".join(terminos) + f" = {suma}")

### 5.3 Los índices de la convolución

Si $x$ empieza en $n_x$ y tiene $L_x$ muestras, y $h$ empieza en $n_h$ y tiene $L_h$ muestras:

$$y \text{ empieza en } n_x + n_h, \qquad \text{y tiene } L_x + L_h - 1 \text{ muestras}$$

La función `pds.convolucion` implementa la fórmula y calcula los índices:

In [ ]:
print(inspect.getsource(pds.convolucion))

In [ ]:
# Ejemplo con índices que NO empiezan en 0
n_x = np.arange(-1, 2)                  # x en n = -1, 0, 1
x = np.array([1.0, 2.0, 1.0])
n_h = np.arange(-2, 1)                  # h en n = -2, -1, 0
h = np.array([1.0, 1.0, 1.0])

n_y, y = pds.convolucion(n_x, x, n_h, h)
print("n_y =", n_y)
print("y   =", y)

figura, ejes = plt.subplots(1, 3, figsize=(14, 3))
pds.graficar_discreta(n_x, x, titulo="x[n]", ax=ejes[0])
pds.graficar_discreta(n_h, h, titulo="h[n]", ax=ejes[1])
pds.graficar_discreta(n_y, y, titulo="y[n] = x[n] * h[n]", ax=ejes[2], color="C3")
for eje in ejes:
    eje.set_xlim(-4, 3)
plt.tight_layout()
plt.show()

### 5.4 Comparación con numpy

`np.convolve(x, h)` calcula **los mismos valores**, pero **no sabe nada de índices**: siempre supone que empiezan en 0.
Los índices los tenemos que calcular nosotros.

In [ ]:
y_numpy = np.convolve(x, h)
pds.comprobar_iguales(y, y_numpy, "pds.convolucion y np.convolve dan los mismos valores")

### 5.5 Convolución = pasar la señal por el sistema

Para un sistema LTI, filtrar con la ecuación en diferencias o convolucionar con $h[n]$ da **lo mismo**:

In [ ]:
x = np.random.randn(30)
b = [0.25, 0.25, 0.25, 0.25]

y_ecuacion = pds.ecuacion_diferencias(b, [1], x)
n_h, h = pds.respuesta_al_impulso(b, [1], N=4)
n_y, y_conv = pds.convolucion(np.arange(len(x)), x, n_h, h)

# La convolución es más larga (L_x + L_h - 1); comparamos las primeras len(x) muestras
pds.comprobar_iguales(y_ecuacion, y_conv[:len(x)], "ecuación en diferencias = convolución con h[n]")

## 6. Estabilidad

Un sistema LTI es estable si $\sum_n |h[n]| < \infty$. Para $h[n] = a^n u[n]$: estable si $|a| < 1$.

In [ ]:
for a in [0.8, -0.95, 1.05]:
    n, h = pds.respuesta_al_impulso(b=[1], a=[1, -a], N=200)
    print(f"a = {a}:  estable = {pds.es_estable(h)}")

## 7. Aplicación: suavizar un ECG con un promedio móvil

El ECG tiene interferencia de 60 Hz. Con $f_s = 360$ Hz, un ciclo de 60 Hz dura exactamente $360/60 = 6$ muestras.
**Un promedio móvil de 6 muestras promedia un ciclo completo de la senoidal, que vale 0**, así que el 60 Hz desaparece.

In [ ]:
fs_ecg = 360
t_ecg, ecg = pds.cargar_senal_texto("../datos/ecg_sintetico.csv", fs=fs_ecg, columna=1)
ecg_filtrado = pds.promedio_movil(ecg, M=6)

inicio = pds.indice_de_muestra(1.5, fs_ecg)
fin = pds.indice_de_muestra(3.0, fs_ecg)
plt.figure(figsize=(11, 3.5))
plt.plot(t_ecg[inicio:fin], ecg[inicio:fin], color="0.7", label="original (con 60 Hz)")
plt.plot(t_ecg[inicio:fin], ecg_filtrado[inicio:fin], "C3", label="promedio móvil M = 6")
plt.legend()
plt.xlabel("t (s)")
plt.title("Filtrado del ECG")
plt.show()

## Resumen

- Un sistema LTI queda descrito por su **respuesta al impulso** $h[n]$.
- La salida para cualquier entrada es la **convolución** $y[n] = \sum_k x[k]\,h[n-k]$.
- La convolución nace de: señal = suma de impulsos + linealidad + invarianza.
- Índices: inicio $n_x + n_h$, longitud $L_x + L_h - 1$.
- FIR: $h$ finita, sin realimentación. IIR: $h$ infinita, con realimentación (puede ser inestable).

En el notebook 10 practicamos la convolución con más detalle (invertir y deslizar, reverberación) y la extendemos a imágenes.

## Comprueba tu comprensión

1. ¿Es lineal $y[n] = x[n] + 1$?
2. Si $x$ tiene 100 muestras y $h$ tiene 20, ¿cuántas tiene $y = x*h$?
3. ¿Cuál es la respuesta al impulso de $y[n] = x[n] - x[n-1]$? ¿Es FIR o IIR?

<details><summary>Ver respuestas</summary>

1. No: con $x = 0$ la salida es 1 ≠ 0 (un sistema lineal debe dar salida cero a entrada cero).
2. $100 + 20 - 1 = 119$.
3. $h[n] = \delta[n] - \delta[n-1] = \{1, -1\}$: FIR.
</details>